<a href="https://colab.research.google.com/github/angelosbc/analise-de-sentimento-steam/blob/main/Treinando_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
import os
import pandas as pd

# 1. Faz o download do seu dataset direto do Kaggle
path = kagglehub.dataset_download("angelosbc/steam-reviews-in-portuguese-pt-br-2021")
print("Pasta do dataset:", path)

# 2. Carrega o CSV filtrado no Pandas
caminho_csv = os.path.join(path, "steam_reviews_brazilian.csv")
df = pd.read_csv(caminho_csv)

# 3. Dá aquela espiada nos dados
print(f"Total de avaliações: {len(df):,}")
df.head()

100%|██████████| 96.6M/96.6M [00:00<00:00, 115MB/s]

Extracting files...


Pasta do dataset: /root/.cache/kagglehub/datasets/angelosbc/steam-reviews-in-portuguese-pt-br-2021/versions/1
Total de avaliações: 918,910


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,29,292030,The Witcher 3: Wild Hunt,85177505,brazilian,Se um dia alguém falar que esse jogo é ruim na...,1611368498,1611368498,True,1,...,True,False,False,76561198844659805,70,4,11115.0,2252.0,11115.0,1.611186e+09
1,30,292030,The Witcher 3: Wild Hunt,85176839,portuguese,bom demais\n,1611367482,1611367482,True,0,...,True,False,False,76561198847379347,4,1,555.0,465.0,555.0,1.611367e+09
2,32,292030,The Witcher 3: Wild Hunt,85176661,brazilian,NaN,1611367193,1611367193,True,0,...,True,False,False,76561198076880796,127,13,875.0,752.0,826.0,1.611370e+09
3,34,292030,The Witcher 3: Wild Hunt,85176249,brazilian,Obra prima!!!,1611366524,1611366524,True,0,...,True,False,False,76561198957873353,32,1,2888.0,1475.0,2888.0,1.611366e+09
4,43,292030,The Witcher 3: Wild Hunt,85173023,brazilian,Jogão da porra.,1611361229,1611361229,True,0,...,True,False,False,76561198141110905,59,4,20193.0,3692.0,20193.0,1.611297e+09


In [2]:
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, roc_auc_score, accuracy_score

# 1. Limpeza rápida e remoção de nulos/ruídos
print("-> Limpando e tratando o texto...")
df = df.dropna(subset=['review', 'recommended'])

# Garantir que o alvo seja 0 ou 1 (booleano ou int)
df['label'] = df['recommended'].astype(int)

def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.lower() # minúsculo
    texto = re.sub(r'http\S+|www\S+|https\S+', '', texto, flags=re.MULTILINE) # remove URLs
    texto = re.sub(r'<.*?>', '', texto) # remove tags HTML
    texto = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', ' ', texto) # mantém letras com acentos e remove símbolos
    texto = re.sub(r'\s+', ' ', texto).strip() # remove espaços duplos
    return texto

df['clean_review'] = df['review'].apply(limpar_texto)
# Filtra reviews que ficaram com menos de 4 caracteres
df = df[df['clean_review'].str.len() >= 4]

print(f"Total de registros válidos pós-limpeza: {len(df):,}")

# 2. Divisão Estratificada (70% Treino, 15% Validação, 15% Teste)
print("-> Dividindo conjuntos (70/15/15)...")
X = df['clean_review']
y = df['label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Treino: {len(X_train):,} | Validação: {len(X_val):,} | Teste: {len(X_test):,}")

# 3. Vetorização TF-IDF (Unigramas + Bigramas, max 5000 features)
print("-> Gerando representação vetorial TF-IDF...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 4. Treinamento da Regressão Logística
print("-> Treinando a Regressão Logística (Baseline 1)...")
modelo_lr = LogisticRegression(max_iter=1000, random_state=42, C=1.0)
modelo_lr.fit(X_train_vec, y_train)

# 5. Avaliação no conjunto de Teste
y_pred = modelo_lr.predict(X_test_vec)
y_prob = modelo_lr.predict_proba(X_test_vec)[:, 1]

print("\n" + "="*40)
print(" RESULTADOS DO BASELINE 1 (TF-IDF + LR)")
print("="*40)
print(f"Acurácia:       {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"ROC-AUC:        {roc_auc_score(y_test, y_prob):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred, target_names=['Negativo (0)', 'Positivo (1)']))

-> Limpando e tratando o texto...
Total de registros válidos pós-limpeza: 853,982
-> Dividindo conjuntos (70/15/15)...
Treino: 597,787 | Validação: 128,097 | Teste: 128,098
-> Gerando representação vetorial TF-IDF...
-> Treinando a Regressão Logística (Baseline 1)...

 RESULTADOS DO BASELINE 1 (TF-IDF + LR)
Acurácia:       0.9618
F1-Score Macro: 0.7899
ROC-AUC:        0.9548

Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

Negativo (0)       0.78      0.49      0.60      7503
Positivo (1)       0.97      0.99      0.98    120595

    accuracy                           0.96    128098
   macro avg       0.87      0.74      0.79    128098
weighted avg       0.96      0.96      0.96    128098



In [3]:
# Treinamento da Regressão Logística com Balanceamento de Classes
print("-> Re-treinando a Regressão Logística com pesos balanceados...")
modelo_lr_balanced = LogisticRegression(
    max_iter=1000,
    random_state=42,
    C=1.0,
    class_weight='balanced'  # <--- O pulo do gato está aqui!
)
modelo_lr_balanced.fit(X_train_vec, y_train)

# Avaliação no conjunto de Teste
y_pred_bal = modelo_lr_balanced.predict(X_test_vec)
y_prob_bal = modelo_lr_balanced.predict_proba(X_test_vec)[:, 1]

print("\n" + "="*45)
print(" RESULTADOS DO BASELINE 1 AJUSTADO (BALANCED)")
print("="*45)
print(f"Acurácia:       {accuracy_score(y_test, y_pred_bal):.4f}")
print(f"F1-Score Macro: {f1_score(y_test, y_pred_bal, average='macro'):.4f}")
print(f"ROC-AUC:        {roc_auc_score(y_test, y_prob_bal):.4f}")
print("\nRelatório de Classificação Detalhado:")
print(classification_report(y_test, y_pred_bal, target_names=['Negativo (0)', 'Positivo (1)']))

-> Re-treinando a Regressão Logística com pesos balanceados...

 RESULTADOS DO BASELINE 1 AJUSTADO (BALANCED)
Acurácia:       0.9012
F1-Score Macro: 0.7279
ROC-AUC:        0.9578

Relatório de Classificação Detalhado:
              precision    recall  f1-score   support

Negativo (0)       0.36      0.88      0.51      7503
Positivo (1)       0.99      0.90      0.95    120595

    accuracy                           0.90    128098
   macro avg       0.68      0.89      0.73    128098
weighted avg       0.95      0.90      0.92    128098

